# STEEX LFM2.5 Fine-Tuning (Kaggle)

Continue fine-tuning from a Colab checkpoint (or start fresh).
Kaggle gives **30 GPU hours/week** with T4 x2.

**Workflow**: Colab trains → pushes to HF Hub → Kaggle pulls and continues

In [ ]:
# Step 1: Install
!pip install -q unsloth[colab-new] datasets huggingface_hub trl

In [ ]:
# Step 2: Auth — use Kaggle secrets for HF token
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

# Or manual login:
# from huggingface_hub import login
# login(token="hf_...")

In [ ]:
# Step 3: Configuration
import torch

HUB_REPO = "YOUR_USERNAME/steex-lfm2-market"  # Same repo as Colab!
RESUME_FROM_HUB = True  # Pull checkpoint from Colab's push

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Step 4: Load model — from Hub (continued training) or base
from unsloth import FastLanguageModel

if RESUME_FROM_HUB:
    # Load the LoRA adapter from previous platform
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=HUB_REPO,  # Pulls your fine-tuned LoRA
        max_seq_length=4096,
        dtype=None,
        load_in_4bit=True,
    )
    print(f"Resumed from {HUB_REPO}")
else:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="LiquidAI/LFM2.5-1.2B-Base",
        max_seq_length=4096,
        dtype=None,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        bias="none",
        use_gradient_checkpointing="unsloth",
    )

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Step 5: Upload or pull dataset
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Option A: Upload via Kaggle dataset
# DATASET_PATH = "/kaggle/input/steex-training-data/train.jsonl"

# Option B: Pull from HF Hub
from huggingface_hub import hf_hub_download
DATASET_PATH = hf_hub_download(
    repo_id=HUB_REPO.replace("steex-lfm2-market", "steex-training-data"),
    filename="train.jsonl",
    repo_type="dataset",
)

tokenizer = get_chat_template(tokenizer, chat_template="chatml")

examples = []
with open(DATASET_PATH) as f:
    for line in f:
        examples.append(json.loads(line))

dataset = Dataset.from_list(examples)

def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(format_example, remove_columns=dataset.column_names)
print(f"Training examples: {len(dataset)}")

In [ ]:
# Step 6: Continue training (more epochs, or on expanded data)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir="./checkpoints",
        num_train_epochs=2,  # Additional epochs on top of Colab's 3
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,  # Lower LR for continued training
        warmup_ratio=0.05,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        save_steps=50,
        logging_steps=10,
        save_total_limit=3,
        fp16=True,
        optim="adamw_8bit",
        seed=42,
        report_to="none",
    ),
    dataset_text_field="text",
    max_seq_length=4096,
    packing=True,
)

trainer.train()

In [ ]:
# Step 7: Push updated model back to Hub
model.push_to_hub(HUB_REPO, tokenizer=tokenizer, private=True)
print(f"Updated model pushed to {HUB_REPO}")

In [ ]:
# Step 8: Export GGUF and push
model.save_pretrained_gguf("steex-lfm2-market", tokenizer, quantization_method="q4_k_m")
model.push_to_hub_gguf(HUB_REPO + "-gguf", tokenizer, quantization_method="q4_k_m", private=True)
print("GGUF exported and pushed!")